# P04 · Notebook 20 · Single-microphone denoiser, N2N after MWF, and smarter scoring (stage 2 of the program)

**Questions, all at −6 dB, on all four machine types:**
1. **One microphone:** does a Noise2Noise denoiser trained on microphone 0 alone beat raw microphone 0? This is exactly the DCASE 2020 setting.
2. **The array:** does an N2N post-filter after MWF beat MWF?
3. **Scoring:** does any of these beat the nearest-clip score (B1)?
   - k nearest neighbours;
   - local density normalisation (TASLP 2026);
   - segment-level scoring, aimed at short events such as valve clicks.

**Rules:** `results/mimii_beats/RULES_denoise_score_written_before_running.md`, written before this run.

**It needs** in Google Drive:
- `P04_BEATs`;
- `P04_TwinFrontEnds` (notebook 19);
- `P04_Encoders` (notebook 15).

A GPU runtime: an L4 is enough, and there is no need for an H100.

**Time:**
- About 67 GB of downloads (the 6 dB and −6 dB zips).
  - In notebook 19 Zenodo gave about 2.4 MB/s per connection, which would be about 8 h for 67 GB.
  - This notebook opens 4 connections per file, which is faster if Zenodo allows it.
- Then about 1.5–2 h of work on an L4.
- Everything is saved to Drive › `P04_DenoiseScore` after each machine type. If Colab disconnects, run all cells again. With background execution on, you can close the browser.

**For later notebooks:** if Drive has room, the audio streams are cached in `P04_AudioCache`, so the next stages do not download again.

**At the end** `p04_n20_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, importlib, threading, math
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR, ENC = MY / "P04_DenoiseScore", Path("/content/work"), MY / "P04_BEATs", MY / "P04_Encoders"
    N19, CACHE = MY / "P04_TwinFrontEnds", MY / "P04_AudioCache"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_DenoiseScore")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR, ENC = Path(os.environ.get("P04_CKPT_DIR", str(OUT))), Path(os.environ.get("P04_ENC_DIR", "P04_Encoders"))
    N19, CACHE = Path(os.environ.get("P04_N19_DIR", "P04_TwinFrontEnds")), Path(os.environ.get("P04_CACHE_DIR", "P04_AudioCache"))
    MY = CACHE.parent
for p in (OUT, WORK, CKPT_DIR, CACHE):
    p.mkdir(parents=True, exist_ok=True)
for p, who in ((N19, "notebook 19"), (ENC, "notebook 15")):
    if not p.exists():
        raise SystemExit(f"missing {p}: {who}'s files must be in Drive")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
T_START = time.time()
print("device:", DEVICE, GPU or "(no GPU: this would be very slow)", "· free disk:", round(shutil.disk_usage(WORK).free / 1e9), "GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · The official BEATs code (the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Every number below was fixed in the rules before this notebook read any clip
RULES = "results/mimii_beats/RULES_denoise_score_written_before_running.md"
PROGRAM = "results/mimii_beats/PROGRAM_full_written_before_running.md"
Z = "https://zenodo.org/records/3384388/files/"
MD5 = {"pump|6dB": "a09ba6060c10fc09cd4c8770213b0b9f", "pump|-6dB": "d20b783a0ff9c93d58f452f98c37b112",
       "fan|6dB": "0890f7d3c2fd8448634e69ff1d66dd47", "fan|-6dB": "f02ae808a58d84b6815b7ec38ff30879",
       "valve|6dB": "fe5fb7c337cd701b1d31dc641e621892", "valve|-6dB": "fdfaf185fea61b21e11952a070a4ada7",
       "slider|6dB": "838c2b3441858359c4704ef13a1b27ff", "slider|-6dB": "49913eda7d37f182cbf8ed5c984140e0"}
ZIPS = {}
for key, h in MD5.items():
    t_, lv_ = key.split("|"); nm = f"{lv_.replace('dB', '_dB')}_{t_}.zip"
    ZIPS[key] = (Z + nm + "?download=1", nm, h)
GB = {"pump": 15.9, "fan": 21.1, "valve": 15.0, "slider": 15.1}     # approximate size of each type's two zips, for the disk check
if TEST:
    _d = json.loads(os.environ["P04_TEST_DATA"]); _d.pop("dcase", None)
    ZIPS = {k: tuple(v) for k, v in _d.items() if k.split("|")[1] in ("6dB", "-6dB")}
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
LEVELS = ["6dB", "-6dB"]                                            # read_all's twins: only these two levels are downloaded
SR, L, H = 16000, 1024, 512                                          # notebook 19's beamformer STFT (MWF weights are per bin of it)
STREAMS = ["CH0_m6", "CH0_6", "MWF_m6", "MWF_6"]                     # microphone 0 and MWF₋₆, applied to the −6 dB clip and its 6 dB twin
BASES = ["CH0", "MWF"]
INPUTS = ["CH0", "N2N-CH0", "MWF", "N2N-MWF"]
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
FOLDS = {1: ["id_00", "id_02"], 2: ["id_04", "id_06"]}
N2N_CFG = dict(n_fft=512, hop=256, hidden=256, layers=2, epochs=60, batch=32, lr=1e-3, lr_end=1e-5, clip=5.0, crop_s=4.0,
               val_every=20, norm_crops=200, in_levels=["-6dB"], seed=0)
BEATS_BATCH, NSEG, KNN, LDN_FLOOR = 16, 10, 4, 1e-6
SCORERS = {"BEATs": ["B1", "kNN4", "LDN1", "SEGmax", "SEGmean"], "ECHO": ["B1", "kNN4", "LDN1"]}
CAND = {"BEATs": ["kNN4", "LDN1", "SEGmax", "SEGmean"], "ECHO": ["kNN4", "LDN1"]}
BAR = 0.01
G0B_MACHINE, G0B_AVG, G0C_COS = 0.02, 0.005, 0.999
PUBLISHED_M6 = {"pump": 0.8977, "fan": 0.8778, "valve": 0.9794, "slider": 0.9441}  # arXiv 2309.07500, Table 1, −6 dB, one microphone
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
if TEST:
    N2N_CFG.update(json.loads(os.environ.get("P04_TEST_CFG", "{}")).get("n2n", {}))
print("rules:", RULES)

In [ ]:
# 5 · Downloads (MD5 against the Zenodo record), a background download for the next type, readers, notebook 11's splits
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


PREFETCH = {}


def prefetch(t):
    # download the next machine type while this one trains its denoisers (only if the disk has room)
    if t is None or t in PREFETCH or (OUT / f"done_{t}.json").exists():
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]




def open2(t):
    return {lv: zipfile.ZipFile(get_file(ZIPS[f"{t}|{lv}"])) for lv in LEVELS}


def read2(zfs, name):
    # one clip at 6 dB and at −6 dB (its twins): level → (samples, 8) float32
    out = {}
    for lv, zf in zfs.items():
        x, sr = read_all(zf.read(name))
        assert sr == SR and x.ndim == 2 and x.shape[1] == 8, (lv, name, sr, x.shape)
        if not TEST:
            assert x.shape[0] == 10 * SR, (lv, name, x.shape)
        out[lv] = x
    return out


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res

In [ ]:
# 6 · The STFT and beamforming of notebook 19 (MWF is applied with its saved weights), and a reconstruction check
SQH = torch.sqrt(torch.hann_window(L, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft(x):
    # (channels, samples) float32 tensor → (channels, bins, frames) complex64; centred, zero padding, square-root Hann
    return torch.stft(x, n_fft=L, hop_length=H, window=SQH, center=True, pad_mode="constant", return_complex=True)


def istft(Y, n):
    return torch.istft(Y, n_fft=L, hop_length=H, window=SQH, center=True, length=n)


def to_dev(x):
    # (samples, channels) numpy → (channels, samples) tensor on the device
    return torch.from_numpy(np.ascontiguousarray(x.T)).to(DEVICE)


def beam(w, X):
    # w: (bins, 8) complex64 tensor, X: (8, bins, frames) → wᴴx: (bins, frames)
    return torch.einsum("km,mkt->kt", w.conj(), X)



_x = (np.random.default_rng(0).standard_normal((3 * SR, 8)) * 0.1).astype(np.float32)
_pr = float(np.abs(istft(stft(to_dev(_x)), len(_x)).cpu().numpy().T - _x).max())
print(f"STFT reconstruction error {_pr:.1e}")
assert _pr < 1e-4

In [ ]:
# 7 · Encoders: BEATs gives a clip embedding and 10 segment embeddings from one pass; ECHO-small gives a clip embedding
@torch.no_grad()
def embed_beats(batch):
    x = batch.to(DEVICE).float() if torch.is_tensor(batch) else torch.from_numpy(np.stack(batch)).to(DEVICE)
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                                   # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
        f = model.patch_embedding(fb.unsqueeze(1))                         # (batch, 768, time patches, 8 frequency patches)
        tp, fp = f.shape[2], f.shape[3]
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)          # tokens in time-major order: index = t·8 + f
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    h = h.float()
    clip = h.mean(1)
    per_t = h.reshape(h.shape[0], tp, fp, h.shape[-1]).mean(2)             # (batch, time patches, 768)
    seg = torch.stack([per_t[:, g].mean(1) for g in np.array_split(np.arange(tp), NSEG)], 1)
    return clip.cpu().numpy(), seg.half().cpu().numpy()


INFO = {}
def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()




@torch.no_grad()
def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class EchoStandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like ECHO's output. Never used on Colab.
    def __init__(self, seed=2, bands=7, dim=16):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def extract_features_from_audio(self, x, sample_rate):
        S = torch.stft(x[0].float(), n_fft=400, hop_length=160, window=torch.hann_window(400, device=x.device), return_complex=True).abs().log1p().T
        e = torch.stack([b.mean(-1) for b in S.chunk(self.bands, dim=-1)], -1)
        q = torch.stack([e[i * e.shape[0] // 8:(i + 1) * e.shape[0] // 8].mean(0) for i in range(8)], 0)
        return torch.einsum("qk,kqd->kd", q, self.P.to(x.device)).flatten(), None


if TEST:
    ECHO = EchoStandIn().to(DEVICE).eval(); INFO["ECHO"] = {"repo": "stand-in (TEST)", "weights_sha256": "", "loader": "stand-in"}
else:
    ECHO = load_echo()
    if INFO["ECHO"]["weights_sha256"] != ECHO_SHA_N15:
        print("NOTE: the ECHO weights differ from notebook 15's; gate G0(e) decides whether its files still apply")

rng = np.random.default_rng(0)
probe = [(rng.standard_normal(10 * SR) * 0.1).astype(np.float32) for _ in range(3)]
eb, es = embed_beats(probe)
with torch.no_grad():
    ref32 = model.extract_features(torch.from_numpy(np.stack(probe)).to(DEVICE))[0].float().mean(1).cpu().numpy()
cos16 = float(np.min(np.sum(eb * ref32, 1) / (np.linalg.norm(eb, axis=1) * np.linalg.norm(ref32, axis=1))))
seg_mean = es.astype(np.float32).mean(1)
print(f"BEATs: clip {eb.shape[1]} numbers, segments {es.shape[1:]}; float16 against float32 cosine ≥ {cos16:.5f}")
assert np.isfinite(eb).all() and np.isfinite(es.astype(np.float32)).all() and cos16 > 0.99, "stop: BEATs output"
assert es.shape[1] == NSEG and np.allclose(seg_mean, eb, atol=0.05 * np.abs(eb).max()), "stop: segments do not average back to the clip"
ee = emb_echo(ECHO, probe)
print(f"ECHO: {ee.shape[1]} numbers per clip · {INFO['ECHO']['loader']}")
assert ee.ndim == 2 and np.isfinite(ee).all()

In [ ]:
# 8 · Scorers (B1, kNN4, LDN1, SEG-max, SEG-mean), the evaluation over notebook 11's splits, and the references of notebook 19
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def knn4(R, X):
    S = unit(X) @ unit(R).T; k = min(KNN, S.shape[1])
    return 1.0 - np.sort(S, axis=1)[:, -k:].mean(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def seg_nn(R3, X3):
    # each test segment's cosine distance to the nearest segment of any reference clip → (test clips, NSEG)
    Ur = torch.from_numpy(unit(R3.reshape(-1, R3.shape[-1]).astype(np.float32))).to(DEVICE)
    Ux = torch.from_numpy(unit(X3.reshape(-1, X3.shape[-1]).astype(np.float32))).to(DEVICE)
    best = torch.cat([(Ux[i:i + 4096] @ Ur.T).max(1).values for i in range(0, len(Ux), 4096)])
    return (1.0 - best).reshape(X3.shape[0], X3.shape[1]).cpu().numpy().astype(np.float64)


def seg_max(R3, X3):
    return seg_nn(R3, X3).max(1)


def seg_mean(R3, X3):
    return seg_nn(R3, X3).mean(1)


SCORE = {"B1": b1, "kNN4": knn4, "LDN1": ldn1, "SEGmax": seg_max, "SEGmean": seg_mean}


def b1_loo(R):
    S = unit(R) @ unit(R).T
    np.fill_diagonal(S, -np.inf)
    return 1.0 - S.max(1)


def fold_of(p):
    return 1 if p in FOLDS[1] else 2


def fixed(E):
    return lambda p, s: E


def folded(E1, E2):
    # Split A: the machine's own fold's denoiser; B and B+n: the other fold's, for every clip involved
    return lambda p, s: (E1 if fold_of(p) == 1 else E2) if s == "A" else (E2 if fold_of(p) == 1 else E1)


def evaluate(get, mid, label, names, scorer=b1, fuse=None):
    # notebook 11's splits. get(p, split) → the embeddings for machine p; fuse: a second getter, B1 rescaled-sum fusion (notebook 15)
    ids = sorted(set(mid.tolist()))
    A = {p: split_a(mid, label, names, p) for p in ids}
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = A[p]; y = label[te]
        fleet = np.concatenate([A[q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            if fuse is None:
                E = get(p, s); sc = scorer(E[ref], E[te])
            else:
                sc = 0.0
                for g in (get, fuse):
                    E = g(p, s); lo = b1_loo(E[ref]); sc = sc + (b1(E[ref], E[te]) - lo.mean()) / lo.std()
            r[s][p] = round(float(roc_auc_score(y, sc)), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


# scorer sanity on a toy set: identical rankings for B1 and kNN with k = 1; LDN1 reduces to B1 when every reference has the same density
_R = np.random.default_rng(3).standard_normal((40, 8)); _X = np.random.default_rng(4).standard_normal((10, 8))
assert np.allclose(b1(_R, _X), 1 - (unit(_X) @ unit(_R).T).max(1))
_Rg = np.vstack([np.eye(8), -np.eye(8)]); _d = ldn1(_Rg, _X); _b = b1(_Rg, _X)
assert np.allclose(_d, _b), "scorer check: LDN1 must equal B1 when every reference has the same local density"
_S3 = np.random.default_rng(5).standard_normal((6, NSEG, 8)); _T3 = np.random.default_rng(6).standard_normal((3, NSEG, 8))
assert seg_max(_S3, _T3).shape == (3,) and np.all(seg_max(_S3, _T3) >= seg_mean(_S3, _T3) - 1e-9), "scorer check: segments"

REF19 = {t: np.load(N19 / f"emb_{t}.npz", allow_pickle=False) for t in TYPES}
REF15 = {m: np.load(ENC / f"enc_{m}_-6dB.npz", allow_pickle=False) for m in MAIN}
N19RES = json.loads((N19 / "twin_front_ends.json").read_text()) if (N19 / "twin_front_ends.json").exists() else None
print("notebook 19's files:", {t: len(REF19[t]["fname"]) for t in TYPES}, "· its decision:", (N19RES or {}).get("decisions", {}).get("program_front_end"))

In [ ]:
# 9 · Noise2Noise post-filter (notebook 19's cell, unchanged): the network, the training loop, inference, and a smoke test on synthetic twins
NF, NH = N2N_CFG["n_fft"], N2N_CFG["hop"]
NB = NF // 2 + 1
SQH2 = torch.sqrt(torch.hann_window(NF, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft2(x):
    return torch.stft(x, n_fft=NF, hop_length=NH, window=SQH2, center=True, pad_mode="constant", return_complex=True)


def istft2(Y, n):
    return torch.istft(Y, n_fft=NF, hop_length=NH, window=SQH2, center=True, length=n)


class N2NNet(nn.Module):
    # log-power per bin → Linear → ReLU → 2-layer bidirectional GRU → Linear → sigmoid: a real mask per bin and frame
    def __init__(self):
        super().__init__()
        self.register_buffer("mu", torch.zeros(NB)); self.register_buffer("sd", torch.ones(NB))
        self.inp = nn.Linear(NB, N2N_CFG["hidden"])
        self.rnn = nn.GRU(N2N_CFG["hidden"], N2N_CFG["hidden"], num_layers=N2N_CFG["layers"], batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * N2N_CFG["hidden"], NB)

    def forward(self, Y):
        f = torch.log10(Y.real ** 2 + Y.imag ** 2 + 1e-10).transpose(1, 2)
        z, _ = self.rnn(torch.relu(self.inp((f - self.mu) / self.sd)))
        return torch.sigmoid(self.out(z)).transpose(1, 2)


def n2n_loss(M, Y, T):
    # per sample Σ|M·Y − T|² / Σ|Y|² (the weight depends on the input only, so the Noise2Noise optimum is unchanged)
    num = ((M * Y - T).abs() ** 2).sum((1, 2)); den = (Y.abs() ** 2).sum((1, 2)).clamp_min(1e-12)
    return (num / den).mean()


def val_loss(net, vx, vt):
    net.eval(); tot = idl = 0.0
    with torch.no_grad():
        for i in range(0, len(vx), 32):
            Y, T = stft2(vx[i:i + 32]), stft2(vt[i:i + 32]); M = net(Y)
            tot += float(n2n_loss(M, Y, T)) * len(Y); idl += float(n2n_loss(torch.ones_like(M), Y, T)) * len(Y)
    net.train()
    return tot / len(vx), idl / len(vx)


def fit(get, n_pairs, norm_x, vx, vt, seed, epochs=None):
    # get(index array, rng) → (inputs, targets) float32 tensors on the device; each epoch is one pass over a fresh permutation
    epochs = epochs or N2N_CFG["epochs"]
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = N2NNet().to(DEVICE)
    with torch.no_grad():
        Fx = torch.cat([torch.log10(stft2(c).abs() ** 2 + 1e-10).permute(0, 2, 1).reshape(-1, NB) for c in norm_x])
        net.mu.copy_(Fx.mean(0)); net.sd.copy_(Fx.std(0).clamp_min(1e-3))
    B = N2N_CFG["batch"]; steps = max(1, math.ceil(n_pairs / B))
    opt = torch.optim.Adam(net.parameters(), lr=N2N_CFG["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps * epochs, eta_min=N2N_CFG["lr_end"])
    v0, vi0 = val_loss(net, vx, vt)
    log = [{"epoch": 0, "train": None, "val": round(v0, 6), "val_identity": round(vi0, 6), "seconds": 0.0}]
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(n_pairs); run = 0.0
        for b in range(steps):
            x, tg = get(order[b * B:(b + 1) * B], rng)
            Y, T = stft2(x), stft2(tg)
            loss = n2n_loss(net(Y), Y, T)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), N2N_CFG["clip"]); opt.step(); sched.step()
            run += float(loss.detach())
        v, vi = val_loss(net, vx, vt)
        log.append({"epoch": ep + 1, "train": round(run / steps, 6), "val": round(v, 6), "val_identity": round(vi, 6), "seconds": round(time.time() - t0, 1)})
    return net, log


@torch.no_grad()
def denoise(net, x):
    # x: (batch, samples) float32 tensor → the post-filtered signal
    Y = stft2(x)
    return istft2(net(Y) * Y, x.shape[1])


# smoke test: harmonic sounds with independent noise in input and target; the denoiser must beat doing nothing on held-out clips
rng = np.random.default_rng(1)
ns, nc = 16000, 96
tt = torch.arange(ns, dtype=torch.float32) / SR
clean = torch.stack([sum(torch.sin(2 * math.pi * float(f0) * k * tt + float(rng.uniform(0, 6))) / k for k in range(1, 9))
                     for f0 in rng.uniform(80, 400, nc)]).to(DEVICE)
nz = lambda g: torch.from_numpy(rng.standard_normal((nc, ns)).astype(np.float32)).to(DEVICE) * g
xin, xtg = clean + nz(2.0), clean + nz(1.0)
net_s, log_s = fit(lambda idx, r: (xin[idx], xtg[idx]), 80, [xin[:16]], xin[80:], xtg[80:], seed=0, epochs=15 if not TEST else 6)
v0, v, vi = log_s[0]["val"], log_s[-1]["val"], log_s[-1]["val_identity"]
print(f"N2N smoke test: validation loss {v0:.3f} untrained → {v:.3f} after {len(log_s) - 1} epochs · {vi:.3f} for doing nothing")
assert v < vi and v < v0, "smoke test: the Noise2Noise denoiser does not learn"
del net_s

In [ ]:
# 10 · One machine type at a time: the four streams, CH0 and MWF embeddings, the audio cache, two Noise2Noise denoisers per fold
STORE = WORK / "stores20"; STORE.mkdir(exist_ok=True)
G0 = {"a": {}, "b": {}, "c": {}}


def to_int16(v):
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def mwf_weights(t):
    D = np.load(N19 / f"stats_{t}.npz", allow_pickle=False)
    return {k.split("__")[0]: D[k] for k in D.files if k.endswith("__w__MWF|-6dB")}


def flush_beats(E, buf, key, force=False):
    if buf.get(key) and (force or len(buf[key]) == BEATS_BATCH):
        c, s = embed_beats(buf[key]); E.setdefault(("BEATs", key), []).append(c); E.setdefault(("SEG", key), []).append(s); buf[key] = []


def build(t, zfs, names, mid, W, embed=True):
    # one pass over every clip: four int16 streams in the local store; CH0 and MWF embeddings at −6 dB (from the float signals)
    n = len(names); ns = read2(zfs, names[0])["6dB"].shape[0]
    mm = {s: np.lib.format.open_memmap(STORE / f"{t}_{s}.npy", mode="w+", dtype=np.int16, shape=(n, ns)) for s in STREAMS}
    Wt = {p: torch.from_numpy(w.astype(np.complex64)).to(DEVICE) for p, w in W.items()}
    E, buf, t0 = {}, {}, time.time()
    for i, x in enumerate(stream(lambda i: read2(zfs, names[i]), range(n))):
        p = mid[i]
        sig = {"CH0_m6": np.ascontiguousarray(x["-6dB"][:, 0]), "CH0_6": np.ascontiguousarray(x["6dB"][:, 0]),
               "MWF_m6": istft(beam(Wt[p], stft(to_dev(x["-6dB"]))), ns).cpu().numpy(),
               "MWF_6": istft(beam(Wt[p], stft(to_dev(x["6dB"]))), ns).cpu().numpy()}
        for s in STREAMS:
            mm[s][i] = to_int16(sig[s])
        if embed:
            for f in BASES:
                buf.setdefault(f, []).append(sig[f"{f}_m6"]); flush_beats(E, buf, f)
                E.setdefault(("ECHO", f), []).append(emb_echo(ECHO, [sig[f"{f}_m6"]]))
        if i % 500 == 0:
            print(f"  {t}: {i}/{n} clips · {time.time() - t0:.0f} s", flush=True)
    for f in list(buf):
        flush_beats(E, buf, f, force=True)
    for s in STREAMS:
        mm[s].flush()
    (STORE / f"{t}.json").write_text(json.dumps({"n": n, "samples": ns}))
    return {k: np.concatenate(v) for k, v in E.items()}


def ensure_store(t, names, mid, W):
    # local store present → use it; else copy from the Drive cache if all four streams are there; else rebuild from the zips
    if (STORE / f"{t}.json").exists() and all((STORE / f"{t}_{s}.npy").exists() for s in STREAMS):
        return {s: np.load(STORE / f"{t}_{s}.npy", mmap_mode="r") for s in STREAMS}
    if all((CACHE / f"{t}_{s}.npy").exists() for s in STREAMS):
        print(f"  {t}: local store copied back from the Drive cache")
        for s in STREAMS:
            shutil.copyfile(CACHE / f"{t}_{s}.npy", STORE / f"{t}_{s}.npy")
        (STORE / f"{t}.json").write_text(json.dumps({"n": len(names), "from": "cache"}))
    else:
        print(f"  {t}: rebuilding the local store from the zips (lost after a disconnect)")
        zfs = open2(t); build(t, zfs, names, mid, W, embed=False)
        for zf in zfs.values():
            zf.close()
    return {s: np.load(STORE / f"{t}_{s}.npy", mmap_mode="r") for s in STREAMS}


def cache_streams(t):
    sizes = {s: (STORE / f"{t}_{s}.npy").stat().st_size for s in STREAMS}
    have = [s for s in STREAMS if (CACHE / f"{t}_{s}.npy").exists() and (CACHE / f"{t}_{s}.npy").stat().st_size == sizes[s]]
    free = shutil.disk_usage(CACHE).free
    need_all = sum(v for s, v in sizes.items() if s not in have)
    need_m6 = sum(sizes[s] for s in ("CH0_m6", "MWF_m6") if s not in have)
    want = [s for s in STREAMS if s not in have] if free >= need_all + 5e9 else \
           [s for s in ("CH0_m6", "MWF_m6") if s not in have] if free >= need_m6 + 5e9 else []
    t0 = time.time()
    for s in want:
        tmp = CACHE / f"{t}_{s}.npy.part"
        try:
            shutil.copyfile(STORE / f"{t}_{s}.npy", tmp); tmp.rename(CACHE / f"{t}_{s}.npy")
        except OSError as e:
            tmp.unlink(missing_ok=True); print(f"  cache write failed for {t} {s}: {repr(e)[:120]}"); break
    done = [s for s in STREAMS if (CACHE / f"{t}_{s}.npy").exists()]
    print(f"  {t}: audio cache in Drive → {done or 'none'} (free {free / 1e9:.0f} GB before · {time.time() - t0:.0f} s)")
    return {"cached": done, "free_gb_before": round(free / 1e9, 1)}


def crops(mm, rows, starts, n):
    return torch.from_numpy(np.stack([mm[r, s:s + n] for r, s in zip(rows, starts)]).astype(np.float32) / 32768.0).to(DEVICE)


def train_n2n(t, base, k, st, mid, label, names):
    # notebook 19's training with input base(x₋₆) and target base(x₆); resumes from Drive
    path = OUT / f"n2n_{t}_{base}_f{k}.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=False)
    trn, va = [], []
    for p in FOLDS[k]:
        if p in set(mid.tolist()):
            tr, _ = split_a(mid, label, names, p)
            v = set(tr[::N2N_CFG["val_every"]].tolist()); va += sorted(v); trn += [i for i in tr if i not in v]
    trn, va = np.array(trn), np.array(va)
    sin, stg = st[f"{base}_m6"], st[f"{base}_6"]
    n, ns = int(N2N_CFG["crop_s"] * SR), sin.shape[1]
    rng = np.random.default_rng(N2N_CFG["seed"] + k)
    nr = rng.choice(trn, N2N_CFG["norm_crops"]); nst = rng.integers(0, ns - n + 1, len(nr))
    norm_x = [crops(sin, nr[i:i + 32], nst[i:i + 32], n) for i in range(0, len(nr), 32)]

    def get(idx, r):
        rows = trn[idx]; sts = r.integers(0, ns - n + 1, len(rows))
        return crops(sin, rows, sts, n), crops(stg, rows, sts, n)

    z0 = np.zeros(len(va), dtype=int)
    vx, vt = crops(sin, va, z0, n), crops(stg, va, z0, n)
    t0 = time.time()
    net, log = fit(get, len(trn), norm_x, vx, vt, seed=N2N_CFG["seed"] + k)
    rec = {"state": {a: b.cpu() for a, b in net.state_dict().items()}, "log": log, "cfg": N2N_CFG, "base": base,
           "train_clips": int(len(trn)), "val_clips": int(len(va)), "seconds": round(time.time() - t0, 1)}
    torch.save(rec, path)
    print(f"  {t} {base} fold {k}: N2N on {len(trn)} clips · validation {log[-1]['val']:.4f} against {log[-1]['val_identity']:.4f}"
          f" for doing nothing · {rec['seconds']:.0f} s", flush=True)
    return rec


def embed_n2n(t, st, recs):
    out = OUT / f"emb_n2n_{t}.npz"
    if out.exists():
        return
    nets = {}
    for (base, k), rec in recs.items():
        net = N2NNet().to(DEVICE); net.load_state_dict(rec["state"]); net.eval(); nets[(base, k)] = net
    E, t0, n = {}, time.time(), st["CH0_m6"].shape[0]
    for i in range(0, n, BEATS_BATCH):
        for base in BASES:
            x = torch.from_numpy(np.asarray(st[f"{base}_m6"][i:i + BEATS_BATCH]).astype(np.float32) / 32768.0).to(DEVICE)
            for k in (1, 2):
                y = denoise(nets[(base, k)], x); c, s = embed_beats(y); key = f"N2N-{base}__{k}"
                E.setdefault(f"BEATs__{key}", []).append(c); E.setdefault(f"SEG__{key}", []).append(s)
                E.setdefault(f"ECHO__{key}", []).append(emb_echo(ECHO, list(y.cpu().numpy())))
    np.savez_compressed(out, **{a: np.concatenate(v) for a, v in E.items()})
    print(f"  {t}: N2N embeddings · {time.time() - t0:.0f} s")


def gate_g0(t, De, mid, label, names):
    R = REF19[t]; worst_m = worst_a = 0.0
    for f in BASES:
        ours = evaluate(fixed(De[f"BEATs__{f}"].astype(np.float64)), mid, label, names)
        ref = evaluate(fixed(R[f"BEATs_m6dB_{f}"].astype(np.float64)), mid, label, names)
        for s in SPLITS:
            worst_a = max(worst_a, abs(ours[s]["avg"] - ref[s]["avg"]))
            worst_m = max(worst_m, max(abs(ours[s][p] - ref[s][p]) for p in set(mid.tolist())))
    G0["b"][t] = {"worst_machine": round(worst_m, 4), "worst_average": round(worst_a, 4)}
    cos = []
    for f in BASES:
        a = De[f"ECHO__{f}"].astype(np.float64)
        b = REF15[t]["ECHO_CH0"] if (f == "CH0" and t in MAIN) else R[f"ECHO_m6dB_{f}"]
        b = b.astype(np.float64); cos.append(float(np.min(np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)))))
    G0["c"][t] = round(min(cos), 6)
    print(f"G0(b) · {t}: BEATs against notebook 19, largest |ΔAUC| per machine {worst_m:.4f} (bar {G0B_MACHINE}), per average {worst_a:.4f} (bar {G0B_AVG})")
    print(f"G0(c) · {t}: ECHO against notebooks 19/15, smallest cosine {G0['c'][t]:.6f} (bar {G0C_COS})")
    if not TEST and (worst_m > G0B_MACHINE or worst_a > G0B_AVG or G0["c"][t] < G0C_COS):
        raise SystemExit("stop: G0(b) or G0(c) failed")


def process_type(t, nxt=None):
    done = OUT / f"done_{t}.json"
    if done.exists():
        print(t, "already done"); return json.loads(done.read_text())
    if t in PREFETCH:
        PREFETCH[t].join()
    t0, times = time.time(), {}
    names, mid, label = REF19[t]["fname"], REF19[t]["mid"], REF19[t]["label"]
    W = mwf_weights(t)
    assert set(W) == set(mid.tolist()), f"{t}: MWF weights for {sorted(W)} but machines {sorted(set(mid.tolist()))}"
    if not (OUT / f"emb_{t}.npz").exists():
        zfs = open2(t); times["download_s"] = round(time.time() - t0, 1)
        for lv, zf in zfs.items():
            zn = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
            G0["a"].setdefault(t, True); G0["a"][t] = G0["a"][t] and bool(np.array_equal(zn, names))
        print(f"G0(a) · {t}: the clips are notebook 19's: {G0['a'][t]}")
        if not G0["a"][t] and not TEST:
            raise SystemExit("stop: G0(a) failed")
        tb = time.time()
        E = build(t, zfs, names, mid, W); times["streams_s"] = round(time.time() - tb, 1)
        np.savez_compressed(OUT / f"emb_{t}.npz", fname=names, mid=mid, label=label,
                            **{f"{enc}__{f}": v.astype(np.float16 if enc == "SEG" else np.float32) for (enc, f), v in E.items()})
        for zf in zfs.values():
            zf.close()
        if IN_COLAB:
            for lv in LEVELS:
                (WORK / ZIPS[f"{t}|{lv}"][1]).unlink(missing_ok=True)
    else:
        G0["a"].setdefault(t, True)
    De = np.load(OUT / f"emb_{t}.npz", allow_pickle=False)
    gate_g0(t, De, mid, label, names)
    st = ensure_store(t, names, mid, W)
    cache = json.loads((OUT / f"cache_{t}.json").read_text()) if (OUT / f"cache_{t}.json").exists() else cache_streams(t)
    (OUT / f"cache_{t}.json").write_text(json.dumps(cache))
    prefetch(nxt)
    tn = time.time()
    recs = {(b, k): train_n2n(t, b, k, st, mid, label, names) for b in BASES for k in (1, 2)}
    embed_n2n(t, st, recs); times["n2n_s"] = round(time.time() - tn, 1)
    res = {"G3": {f"{b}|{k}": bool(r["log"][-1]["val"] < r["log"][-1]["val_identity"]) for (b, k), r in recs.items()},
           "logs": {f"{b}|{k}": r["log"] for (b, k), r in recs.items()}, "train_clips": {f"{b}|{k}": r["train_clips"] for (b, k), r in recs.items()},
           "cache": cache, "times": times, "minutes": round((time.time() - t0) / 60, 1), "G0": {g: G0[g].get(t) for g in G0}}
    done.write_text(json.dumps(res))
    if IN_COLAB:
        for s in STREAMS:
            (STORE / f"{t}_{s}.npy").unlink(missing_ok=True)
        (STORE / f"{t}.json").unlink(missing_ok=True)
    print(f"{t}: done in {res['minutes']} min")
    return res

In [ ]:
# 11 · Run the four machine types. Safe to re-run after a disconnect: finished parts are skipped
SUMMARY = {}
for j, t in enumerate(TYPES):
    SUMMARY[t] = process_type(t, nxt=TYPES[j + 1] if j + 1 < len(TYPES) else None)
for t in TYPES:                                                   # gates of types finished in an earlier session
    for g in G0:
        if t not in G0[g] and SUMMARY[t]["G0"].get(g) is not None:
            G0[g][t] = SUMMARY[t]["G0"][g]
print(f"all four types done · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 12 · Scores for every input × encoder × scorer, and decisions D1–D3 exactly as written in the rules
EMB, META = {}, {}
for t in TYPES:
    D = np.load(OUT / f"emb_{t}.npz", allow_pickle=False); META[t] = (D["fname"], D["mid"], D["label"])
    for f in BASES:
        for enc in ("BEATs", "ECHO", "SEG"):
            EMB[(enc, t, f)] = D[f"{enc}__{f}"] if enc == "SEG" else D[f"{enc}__{f}"].astype(np.float64)   # segments stay float16 in memory
    Dn = np.load(OUT / f"emb_n2n_{t}.npz", allow_pickle=False)
    for f in BASES:
        for enc in ("BEATs", "ECHO", "SEG"):
            EMB[(enc, t, f"N2N-{f}")] = tuple(Dn[f"{enc}__N2N-{f}__{k}"] if enc == "SEG" else Dn[f"{enc}__N2N-{f}__{k}"].astype(np.float64) for k in (1, 2))


def getter(key):
    E = EMB[key]
    return folded(*E) if isinstance(E, tuple) else fixed(E)


RES = {}
t_sc = time.time()
for t in TYPES:
    fn, mid, label = META[t]
    for f in INPUTS:
        for enc, scs in SCORERS.items():
            for sc in scs:
                src_enc = "SEG" if sc.startswith("SEG") else enc
                RES[(enc, t, f, sc)] = evaluate(getter((src_enc, t, f)), mid, label, fn, SCORE[sc])
FUSED = {(t, f): evaluate(getter(("BEATs", t, f)), META[t][1], META[t][2], META[t][0], fuse=getter(("ECHO", t, f))) for t in TYPES for f in INPUTS}
print(f"scored in {time.time() - t_sc:.0f} s")


def S(enc, f, sc, types):
    return round(float(np.mean([(RES[(enc, t, f, sc)]["A"]["avg"] + RES[(enc, t, f, sc)]["B+10"]["avg"]) / 2 for t in types])), 4)


def S2(f, types):
    return round((S("BEATs", f, "B1", types) + S("ECHO", f, "B1", types)) / 2, 4)


g3 = {b: all(SUMMARY[t]["G3"][f"{b}|{k}"] for t in TYPES for k in (1, 2)) for b in BASES}
DEC = {}
for d, (cand, ref) in (("D1", ("N2N-CH0", "CH0")), ("D2", ("N2N-MWF", "MWF"))):
    dev, dev_r, conf, conf_r = S2(cand, MAIN), S2(ref, MAIN), S2(cand, AUX), S2(ref, AUX)
    ok = g3[ref] and dev >= dev_r + BAR and conf >= conf_r - BAR
    DEC[d] = {"pass": bool(ok), "candidate": cand, "dev": dev, "dev_ref": dev_r, "bar": round(dev_r + BAR, 4), "conf": conf, "conf_ref": conf_r,
              "conf_bar": round(conf_r - BAR, 4), "G3_all_types": bool(g3[ref]), "chosen": cand if ok else ref}
IN1, IN2 = DEC["D1"]["chosen"], DEC["D2"]["chosen"]
for enc in SCORERS:
    sdev = {sc: round((S(enc, IN1, sc, MAIN) + S(enc, IN2, sc, MAIN)) / 2, 4) for sc in SCORERS[enc]}
    sconf = {sc: round((S(enc, IN1, sc, AUX) + S(enc, IN2, sc, AUX)) / 2, 4) for sc in SCORERS[enc]}
    best = max(CAND[enc], key=lambda sc: (sdev[sc], -CAND[enc].index(sc)))
    ok = sdev[best] >= sdev["B1"] + BAR and sconf[best] >= sconf["B1"] - BAR
    DEC[f"D3|{enc}"] = {"pass": bool(ok), "best": best, "dev": sdev, "conf": sconf, "bar": round(sdev["B1"] + BAR, 4),
                        "conf_bar": round(sconf["B1"] - BAR, 4), "chosen": best if ok else "B1"}
SC = {enc: DEC[f"D3|{enc}"]["chosen"] for enc in SCORERS}

print(f"{'type':7s} {'input':8s} {'enc':6s} {'scorer':8s} " + " ".join(f"{s:>7s}" for s in SPLITS))
for (enc, t, f, sc), r in sorted(RES.items(), key=lambda kv: (TYPES.index(kv[0][1]), INPUTS.index(kv[0][2]), kv[0][0], SCORERS[kv[0][0]].index(kv[0][3]))):
    print(f"{t:7s} {f:8s} {enc:6s} {sc:8s} " + " ".join(f"{r[s]['avg']:7.4f}" for s in SPLITS))
print()
for d in ("D1", "D2"):
    v = DEC[d]
    print(f"{d} · {v['candidate']}: development {v['dev']} against {v['dev_ref']} + {BAR} · confirmation {v['conf']} against {v['conf_ref']} − {BAR}"
          f" · G3 {v['G3_all_types']} →", "pass" if v["pass"] else "fail", "→ input:", v["chosen"])
for enc in SCORERS:
    v = DEC[f"D3|{enc}"]
    print(f"D3 · {enc}: development {v['dev']} · confirmation {v['conf']} · best {v['best']} →", "pass" if v["pass"] else "fail", "→ scorer:", v["chosen"])
print(f"→ single-microphone input {IN1}, array input {IN2}, scorers {SC}")

REP = {"single_mic_vs_published": {}, "n2n_mvdr_n19": {}, "fused_B1": {}}
for t in TYPES:
    REP["single_mic_vs_published"][t] = {"published": PUBLISHED_M6[t], "CH0 BEATs B1": RES[("BEATs", t, "CH0", "B1")]["A"]["avg"],
                                         **{f"{IN1} {enc} {SC[enc]}": RES[(enc, t, IN1, SC[enc])]["A"]["avg"] for enc in SCORERS}}
    REP["fused_B1"][t] = {f: {s: FUSED[(t, f)][s]["avg"] for s in ("A", "B+10")} for f in INPUTS}
    if N19RES:
        REP["n2n_mvdr_n19"][t] = {enc: {s: N19RES["results"][f"{enc}|{t}|-6dB|N2N"][s]["avg"] for s in ("A", "B+10")} for enc in ("BEATs", "ECHO")}
print("\nreported:", json.dumps(REP, indent=1))

In [ ]:
# 13 · Save, two pictures, and one zip to download
import matplotlib
import matplotlib.pyplot as plt

RESULT = {"rules": RULES, "program": PROGRAM, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "torch": torch.__version__, "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                   "echo": INFO.get("ECHO"), "zips": {k: v[1:] for k, v in ZIPS.items()}},
          "G0": G0, "types": SUMMARY, "decisions": DEC, "inputs": {"single_mic": IN1, "array": IN2}, "scorers": SC, "reported": REP,
          "results": {"|".join(k): v for k, v in RES.items()}, "fused": {"|".join(k): v for k, v in FUSED.items()}}
(OUT / "denoise_score.json").write_text(json.dumps(RESULT, indent=1, default=float))

COL = {"CH0": "#8d8d8d", "N2N-CH0": "#2a78d6", "MWF": "#eb6834", "N2N-MWF": "#1baf7a"}
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey="row")
for c, t in enumerate(TYPES):
    for r, s in enumerate(("A", "B+10")):
        ax = axes[r, c]; x = np.arange(2)
        for j, f in enumerate(INPUTS):
            v = [RES[(enc, t, f, "B1")][s]["avg"] for enc in ("BEATs", "ECHO")]
            ax.bar(x + (j - 1.5) * 0.2, v, 0.18, color=COL[f], label=f if (r == 0 and c == 0) else None)
        ax.set_xticks(x); ax.set_xticklabels(["BEATs", "ECHO"]); ax.set_ylim(0.4, 1.0)
        ax.grid(axis="y", color="#e6e6e3", lw=0.8); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"{t} · {'known machine (A)' if s == 'A' else 'new machine, 10 clips (B+10)'}", fontsize=10)
        if c == 0:
            ax.set_ylabel("average AUC at −6 dB (B1)")
fig.legend(loc="lower center", ncol=4, frameon=False)
fig.suptitle(f"Notebook 20 · inputs at −6 dB · single-microphone: {IN1} · array: {IN2}", fontsize=12)
fig.tight_layout(rect=(0, 0.05, 1, 0.95)); fig.savefig(OUT / "denoise_score_inputs.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, enc in zip(axes, SCORERS):
    v = DEC[f"D3|{enc}"]; scs = SCORERS[enc]; x = np.arange(len(scs))
    ax.bar(x - 0.18, [v["dev"][sc] for sc in scs], 0.34, color="#2a78d6", label="development (pumps, fans)" if enc == "BEATs" else None)
    ax.bar(x + 0.18, [v["conf"][sc] for sc in scs], 0.34, color="#eb6834", label="confirmation (valves, slide rails)" if enc == "BEATs" else None)
    ax.axhline(v["bar"], color="#2a78d6", ls="--", lw=0.8, label="development bar (B1 + 0.01)" if enc == "BEATs" else None)
    ax.set_xticks(x); ax.set_xticklabels(scs); ax.set_ylim(0.5, 1.0); ax.set_title(f"{enc} · scorer on the chosen inputs · chosen: {SC[enc]}", fontsize=10)
    ax.grid(axis="y", color="#e6e6e3", lw=0.8); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("score S = mean of (A + B+10)/2")
fig.legend(loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(OUT / "denoise_score_scorers.png", dpi=150); plt.show()

ZIP = OUT / "p04_n20_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("denoise_score.json", "denoise_score_inputs.png", "denoise_score_scorers.png"):
        z.write(OUT / f, f)
print("saved", ZIP, f"· {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n20_results.zip`). It is also in Drive › `P04_DenoiseScore`. The large files (embeddings and denoisers) stay in Drive, and so does the audio cache in `P04_AudioCache`.